# Peyk SDK demo

Notebook version of [`demo.py`](demo.py) — the same end-to-end flow (configure the pipeline,
start/wait on whichever vLLM sidecars are needed, run the main container over sample input),
broken into cells so each step's result is inspectable before moving to the next.

Run `uv sync --extra notebook` from the repo root once first, then select the repo-level
`.venv` as this notebook's kernel. Requires Docker with GPU support, and the `peyk:dev` image
already built (or run the `build_image` cell below).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
sys.path.insert(0, str(REPO_ROOT / "sdk" / "src"))

from peyk import Peyk, PipelineConfig, StageConfig

IMAGE = "peyk:dev"
INPUT_DIR = REPO_ROOT / "hotstorage" / "input"
OUTPUT_DIR = REPO_ROOT / "hotstorage" / "output"
CONFIG_DIR = REPO_ROOT / "hotstorage" / "peyk-config"

peyk = Peyk(image=IMAGE)

## (Optional) Build the image

Skip this cell if `peyk:dev` is already built.

In [ ]:
# peyk.build_image(REPO_ROOT / "containers" / "peyk", tag=IMAGE)

## Configure the pipeline

Same defaults as `containers/peyk/stages/orchestrator/config/example.yaml`. Edit the
`StageConfig(...)` calls below to try different models — see that file's comments for every
job's available options.

In [ ]:
config = PipelineConfig(
    layout=StageConfig(model="heron"),
    tsr=StageConfig(model="surya"),
    ocr=StageConfig(model="paddleocr-vl", lang="arabic"),
    force_scanned=True,
    cell_ocr=StageConfig(model="surya", lang="arabic"),
    figures=StageConfig(model="gemini-3-1-flash-lite")
)

peyk.configure(config, config_dir=CONFIG_DIR)
print(config.to_yaml())

layout:
  model: heron
tsr:
  model: claude-sonnet-4-5
ocr:
  model: gemini-3-1-flash-lite
  lang: arabic
figures:
  model: gemini-3-1-flash-lite
surya_smart_table_split:
  enabled: true
  sharpness_threshold_laplacian_var: 450
  pixel_safety_margin: 0.97
  max_upscale_cap: 2.0
  min_scale: 1.0
born_digital:
  min_chars_per_page: 20
  force_scanned: true
cell_ocr:
  model: claude-sonnet-4-5
  lang: arabic



## Credentials

Picked up automatically from `containers/peyk/.env` (`AWS_BEARER_TOKEN_BEDROCK`) and
`containers/peyk/gcp-key.json` if present — only needed if the config above selects a `vlm`
backend (e.g. `figures: gemini-3-1-flash-lite` does). Override either argument directly if you
keep credentials elsewhere.

In [3]:
bedrock_env = REPO_ROOT / "containers" / "peyk" / ".env"
bedrock_token = None
if bedrock_env.exists():
    for line in bedrock_env.read_text().splitlines():
        if line.startswith("AWS_BEARER_TOKEN_BEDROCK="):
            bedrock_token = line.split("=", 1)[1].strip()

gcp_key = REPO_ROOT / "containers" / "peyk" / "gcp-key.json"
gcp_key = gcp_key if gcp_key.exists() else None

peyk.set_credentials(bedrock_bearer_token=bedrock_token, gcp_key_path=gcp_key)

## Start the sidecars this config needs

Only starts (and waits on) whichever of `peyk-vllm-surya`/`peyk-vllm-paddleocr` the config
above actually resolves to. Surya's own cold start is documented at ~14 minutes — this cell
blocks until whatever's needed reports ready.

In [4]:
needed = peyk.ensure_sidecars(wait=True)
print(f"Sidecars ready: {needed or '(none needed for this config)'}")

Sidecars ready: (none needed for this config)


## Run the pipeline

Drop a PDF into `hotstorage/input/` first (or point `INPUT_DIR` above at `data/` for one of the
repo's sample PDFs).

In [5]:
result = peyk.run(input_dir=INPUT_DIR, output_dir=OUTPUT_DIR, stream_logs=True)
print(result.logs)
print(f"Exit code: {result.exit_code}")

[peyk-orchestrator] job_id=b05c506c5419408d8b7afc2b6fc5a25b
[peyk-orchestrator] running peyk-layout over input batch...
@@PEYK-EVENT@@{"ts": 1788277860.5187616, "job_id": "b05c506c5419408d8b7afc2b6fc5a25b", "stage": "pipeline", "event": "job_start", "input_dir": "/hotstorage/input", "output_dir": "/hotstorage/output"}
@@PEYK-EVENT@@{"ts": 1788277860.5196645, "job_id": "b05c506c5419408d8b7afc2b6fc5a25b", "stage": "layout", "event": "dispatch_start", "model": "heron", "input_dir": "/hotstorage/input", "output_dir": "/hotstorage/workdir/layout_out"}
[peyk-orchestrator] dispatching --stage layout --model heron --input /hotstorage/input --output /hotstorage/workdir/layout_out --visualize
[peyk-layout] loading backend 'heron'...
Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 3334.55it/s]
[peyk-layout] backend 'heron' loaded.
[peyk-layout] processing _الادوات المالية 1.pdf...
[peyk-layout] wrote /hotstorage/workdir/layout_out/_الادوات المالية 1.json
[peyk-layout] processing bdc_sample_t

## Job history

Every `run()` call is recorded automatically — a job row plus one event per pipeline-stage
dispatch (start/end, duration, exit code, and any stub-fallback/credential error), regardless of
anything below. `result.job_id` identifies this run; `peyk.jobs` is queryable for any past run
too (`peyk.jobs.list_jobs()`). See `docs-personal/central_logging_system.md` /
`sdk/README.md`'s "Job history & artifacts" section.

In [6]:
job = peyk.jobs.get_job(result.job_id)
print(f"job {job.job_id}: {job.status} (exit {job.exit_code})")

for event in peyk.jobs.get_events(result.job_id):
    detail = f"{event.duration_s:.2f}s" if event.duration_s is not None else (event.message or "")
    print(f"  [{event.stage or '?':>10}] {event.event or '?':<14} {detail}")
    if event.artifact_path:
        print(f"  {'':>10}   artifacts: {event.artifact_path}")

job b05c506c5419408d8b7afc2b6fc5a25b: succeeded (exit 0)
  [  pipeline] job_start      
  [    layout] dispatch_start 
  [    layout] dispatch_end   31.59s
  [table_full] dispatch_start 
  [table_full] dispatch_end   85.57s
  [       ocr] dispatch_start 
  [       ocr] dispatch_end   100.72s
  [   figures] dispatch_start 
  [   figures] dispatch_end   6.42s
  [  pipeline] job_end        


## (Optional) Persist this run's crops/model output for debugging

Off by default (a single job's table-cell OCR crops alone can be hundreds of files) — pass
`persist_artifacts=True` to copy each dispatched stage's own input/output directory (crops,
per-region model JSON/HTML, viz PNGs) out of the shared workdir volume into a local,
stage-partitioned tree (`peyk.artifacts`, default `~/.peyk/artifacts/<stage>/<job_id>/...`).
Re-runs the pipeline — only run this cell if you want a second pass with artifacts kept.

In [7]:
artifact_result = peyk.run(input_dir=INPUT_DIR, output_dir=OUTPUT_DIR, persist_artifacts=True)
print(f"job {artifact_result.job_id}: exit {artifact_result.exit_code}")

for event in peyk.jobs.get_events(artifact_result.job_id, event="dispatch_end"):
    print(f"  [{event.stage}] {event.artifact_path}")

job 9d14b6d36bf5420c95020d3c44e40c89: exit 0
  [layout] C:\Users\KhaledIbrahim\.peyk\artifacts\layout\9d14b6d36bf5420c95020d3c44e40c89\layout_out
  [table_full] C:\Users\KhaledIbrahim\.peyk\artifacts\table_full\9d14b6d36bf5420c95020d3c44e40c89\table_full_in;C:\Users\KhaledIbrahim\.peyk\artifacts\table_full\9d14b6d36bf5420c95020d3c44e40c89\table_full_out
  [ocr] C:\Users\KhaledIbrahim\.peyk\artifacts\ocr\9d14b6d36bf5420c95020d3c44e40c89\ocr_in;C:\Users\KhaledIbrahim\.peyk\artifacts\ocr\9d14b6d36bf5420c95020d3c44e40c89\ocr_out
  [figures] C:\Users\KhaledIbrahim\.peyk\artifacts\figures\9d14b6d36bf5420c95020d3c44e40c89\figures_in;C:\Users\KhaledIbrahim\.peyk\artifacts\figures\9d14b6d36bf5420c95020d3c44e40c89\figures_out


In [ ]:
# Clean up persisted artifacts later, independently of the job/event history (which stays
# queryable in peyk.jobs either way):
# peyk.artifacts.cleanup(job_id=artifact_result.job_id)   # this job, every stage
# peyk.artifacts.cleanup(stage="ocr")                     # every job's ocr artifacts

## (Optional) Stop the sidecars

In [ ]:
# peyk.stop_sidecars()